# IEEE ISAIA 2026: Beyond the Chatbot — Live Execution Harness
**Paper Title:** Beyond the Chatbot: Architecture, Autonomy, and Inference Economics  
**Authors:** Satya Prakash, Rohan Shahane, Dr. Avimanyou Vatsa (DC Lab, Fairleigh Dickinson University)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/psatyaa/isaia-agent-economics/blob/main/demo_notebook.ipynb)

---
### Case Study Objective:
An enterprise billing alert has detected an unbudgeted **+38% cloud cost jump**. The agent is tasked with diagnosing the runaway resource group and generating remediation.

This notebook supports **two execution modes**:
1. **Replay Mode (0-Key / Default):** Uses deterministic recorded execution traces (100% offline, guaranteed instant execution).
2. **Live Mode (Real LLM Execution):** Calls real Groq inference endpoints using `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` to measure live tokens.

In [ ]:
#@title 1. Environment Setup & Execution Mode Selector
#@markdown Choose whether to run live against the Groq API or run instant deterministic replay.
EXECUTION_MODE = "Replay (0-Key Offline)" #@param ["Replay (0-Key Offline)", "Live (Groq API)"]
GROQ_API_KEY = "" #@param {type:"string"}

!pip install -q tabulate matplotlib groq

# Clone repository code if running standalone in Google Colab
import os, sys
if not os.path.exists('src'):
    !git clone https://github.com/psatyaa/isaia-agent-economics.git
    sys.path.insert(0, 'isaia-agent-economics')
else:
    sys.path.insert(0, '.')

from src.controls import GovernanceControls
from src.agent_loop import AgentHarness
from tabulate import tabulate
import matplotlib.pyplot as plt

# Resolve execution mode & API key
mode_str = "live" if "Live" in EXECUTION_MODE else "replay"
api_key_val = GROQ_API_KEY.strip() or os.environ.get("GROQ_API_KEY")

if mode_str == "live" and not api_key_val:
    print("⚠️ Live mode selected but GROQ_API_KEY is empty. Automatically switching to REPLAY mode.")
    mode_str = "replay"

print(f"✅ Execution Mode Active: {mode_str.upper()}")
if mode_str == "live":
    print("   Using Groq API: llama-3.3-70b-versatile (Frontier) & llama-3.1-8b-instant (SLM)")
else:
    print("   Using Deterministic Replay Traces (Slide 7 vs Slides 8 & 9)")

In [ ]:
#@title 2. Run A: Unconstrained Autonomous Execution (Slide 7 Replication)
print('=' * 80)
print(f"RUN A: UNCONSTRAINED AUTONOMOUS AGENT LOOP [{mode_str.upper()} MODE]")
print('Goal: Query billing API for sub-prod-analytics-01 and isolate cost anomaly.')
print('Configuration: Frontier Model (70B) every step | Budgets: OFF | Compactor: OFF')
print('=' * 80)

# Run unconstrained harness
ctrls_a = GovernanceControls(enable_budget=False, enable_routing=False, enable_compaction=False)
harness_a = AgentHarness(ctrls_a, mode=mode_str, api_key=api_key_val)
result_a = harness_a.run_unconstrained()

table_a = []
for t in result_a['turns']:
    table_a.append([
        f"Turn {t['turn']}",
        t['phase'],
        t['model_tier'].upper(),
        f"{t['in_tokens']:,} toks",
        f"{t['cum_in_tokens']:,} toks",
        f"${t['turn_cost_usd']:.4f}",
        t['status']
    ])

headers = ['Turn', 'Phase', 'Tier', 'Input Tokens', 'Accumulated In', 'Turn Cost', 'Status']
print(tabulate(table_a, headers=headers, tablefmt='fancy_grid'))

print(f"\nFINAL WORKLOAD: {result_a['total_in_tokens']:,} Input Tokens | Cost: ${result_a['cost_usd']:.4f}")
print(f"STATUS: {result_a['status']}")

In [ ]:
#@title 3. Run B: Governed Execution with Four Controls (Slides 8 & 9 Replication)
print('=' * 80)
print(f"RUN B: GOVERNED AGENT LOOP (FOUR CONTROLS ACTIVE) [{mode_str.upper()} MODE]")
print('Controls: max_iterations=3 | SLM 8B Routing | Context Compaction | Frozen Prefix')
print('=' * 80)

ctrls_b = GovernanceControls(enable_budget=True, max_iterations=3, enable_routing=True, enable_compaction=True)
harness_b = AgentHarness(ctrls_b, mode=mode_str, api_key=api_key_val)
result_b = harness_b.run_governed()

table_b = []
for t in result_b['turns']:
    table_b.append([
        f"Turn {t['turn']}",
        t['phase'],
        t['model_tier'].upper(),
        f"{t['in_tokens']:,} toks",
        f"{t['cum_in_tokens']:,} toks",
        f"${t['turn_cost_usd']:.4f}",
        t['status']
    ])

headers = ['Turn', 'Phase', 'Tier', 'Input Tokens', 'Accumulated In', 'Turn Cost', 'Status']
print(tabulate(table_b, headers=headers, tablefmt='fancy_grid'))
print(f"\nFINAL WORKLOAD: {result_b['total_in_tokens']:,} Input Tokens | Cost: ${result_b['cost_usd']:.4f}")
print(f"STATUS: {result_b['status']}")
print(result_b['handoff_card'])

In [ ]:
#@title 4. Side-by-Side Inference Economics Visualizer (Slide 10 Bridge)
tokens_a = result_a['total_in_tokens']
tokens_b = result_b['total_in_tokens']
savings_pct = ((tokens_a - tokens_b) / tokens_a) * 100 if tokens_a > 0 else 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Token Burn Comparison
bars1 = ax1.bar(['Run A: Unconstrained', 'Run B: Governed'], [tokens_a, tokens_b], color=['#7A1835', '#14283D'], width=0.5)
ax1.set_title('Cumulative Input Token Consumption', fontsize=13, fontweight='bold', color='#14283D')
ax1.set_ylabel('Input Tokens', fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + (max(tokens_a, 1000) * 0.02), f'{int(yval):,} toks', ha='center', va='bottom', fontweight='bold')

# Dollar Cost Comparison
cost_a = result_a['cost_usd']
cost_b = result_b['cost_usd']
bars2 = ax2.bar(['Run A: Unconstrained', 'Run B: Governed'], [cost_a, cost_b], color=['#C9982D', '#14283D'], width=0.5)
ax2.set_title('Inference Compute Spend (USD)', fontsize=13, fontweight='bold', color='#14283D')
ax2.set_ylabel('Cost ($ USD)', fontsize=11)
ax2.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + (max(cost_a, 0.001) * 0.02), f'${yval:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle(f'Controlled Exit Yields {savings_pct:.1f}% Reduction in Computational Waste', fontsize=15, fontweight='bold', color='#7A1835')
plt.tight_layout()
plt.show()

print('=' * 80)
print('UNIT ECONOMICS SUMMARY:')
print(f'• Compute Waste Eliminated: {tokens_a - tokens_b:,} tokens (-{savings_pct:.1f}% reduction)')
print(f'• Run A Outcome: UNRESOLVED ($0.00 value per ${cost_a:.4f} spend)')
print(f'• Run B Outcome: CONTROLLED HANDOFF (${cost_b:.4f} spend enables human resolution)')
print('=' * 80)